# 03 Data Utils Pipeline

Notebook nay flatten phan chuan bi du lieu trong `src/data_utils.py` thanh tung cell de de theo doi va chinh sua.

File `src/data_utils.py` duoc giu nguyen de tranh lam gay import va test hien tai.


In [ ]:
import random
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [ ]:
cfg = SimpleNamespace(
    data_path=Path("../data/processed/data2225_done.csv"),
    target="PM25",
    time_col="Local Time",
    lookback=336,
    horizon=72,
    target_transform="log",  # "log", "sqrt", "none"
    train_end="2023-12-31 23:00:00",
    val_start="2024-01-01 00:00:00",
    val_end="2024-12-31 23:00:00",
    test_start="2025-01-01 00:00:00",
    freq="1h",
    seed=42,
)

TARGET_FEATURE_LAGS = (1, 3, 6, 12, 24, 48, 72, 168)
TARGET_FEATURE_WINDOWS = (24, 72, 168)
TARGET_FEATURE_HISTORY = max(max(TARGET_FEATURE_LAGS), max(TARGET_FEATURE_WINDOWS))

cfg


In [ ]:
np.random.seed(cfg.seed)
random.seed(cfg.seed)
tf.random.set_seed(cfg.seed)


In [ ]:
df = pd.read_csv(cfg.data_path)

if cfg.time_col not in df.columns:
    raise KeyError(f"Khong tim thay cot thoi gian '{cfg.time_col}'.")
if cfg.target not in df.columns:
    raise KeyError(f"Khong tim thay cot target '{cfg.target}'.")

df[cfg.time_col] = pd.to_datetime(df[cfg.time_col], errors="coerce")
df = df.dropna(subset=[cfg.time_col])
df = df.set_index(cfg.time_col).sort_index()
df = df[~df.index.duplicated(keep="last")]
df = df.asfreq(cfg.freq)

print("Loaded df:", df.shape)
df.head()


In [ ]:
raw_train_df = df.loc[: cfg.train_end].copy()
raw_val_df = df.loc[cfg.val_start : cfg.val_end].copy()
raw_test_df = df.loc[cfg.test_start :].copy()

if raw_train_df.empty:
    raise ValueError("Train set rong.")
if raw_val_df.empty:
    raise ValueError("Validation set rong.")
if raw_test_df.empty:
    print("Canh bao: Test set rong.")

print("Raw train:", raw_train_df.shape)
print("Raw val  :", raw_val_df.shape)
print("Raw test :", raw_test_df.shape)


In [ ]:
history_len = cfg.lookback + TARGET_FEATURE_HISTORY

raw_sections = {
    "train": {
        "history": raw_train_df.iloc[0:0].copy(),
        "current": raw_train_df.copy(),
    },
    "val_full": {
        "history": raw_train_df.copy(),
        "current": raw_val_df.copy(),
    },
    "test_full": {
        "history": pd.concat([raw_train_df, raw_val_df], axis=0),
        "current": raw_test_df.copy(),
    },
}

processed_sections = {}

for name, section in raw_sections.items():
    history_df = section["history"].tail(history_len)
    current_df = section["current"].copy()
    working_df = pd.concat([history_df, current_df], axis=0) if not history_df.empty else current_df.copy()

    num_cols = working_df.select_dtypes(include=[np.number]).columns.tolist()
    if num_cols:
        working_df[num_cols] = working_df[num_cols].interpolate(method="time").ffill().bfill()

    if "IsHoliday" in working_df.columns:
        holiday = pd.to_numeric(working_df["IsHoliday"], errors="coerce").ffill().bfill().fillna(0)
        working_df["IsHoliday"] = holiday.astype(int)

    obj_cols = working_df.select_dtypes(include=["object", "category"]).columns.tolist()
    for col in obj_cols:
        working_df[col] = working_df[col].ffill().bfill()

    idx = working_df.index
    working_df["day_of_week"] = idx.dayofweek
    working_df["month"] = idx.month
    working_df["hour"] = idx.hour
    working_df["is_weekend"] = (idx.dayofweek >= 5).astype(int)
    working_df["hour_sin"] = np.sin(2 * np.pi * idx.hour / 24)
    working_df["hour_cos"] = np.cos(2 * np.pi * idx.hour / 24)
    working_df["dow_sin"] = np.sin(2 * np.pi * idx.dayofweek / 7)
    working_df["dow_cos"] = np.cos(2 * np.pi * idx.dayofweek / 7)
    working_df["month_sin"] = np.sin(2 * np.pi * idx.month / 12)
    working_df["month_cos"] = np.cos(2 * np.pi * idx.month / 12)

    active_lags = [lag for lag in TARGET_FEATURE_LAGS if lag < len(working_df)]
    active_windows = [window for window in TARGET_FEATURE_WINDOWS if window < len(working_df)]
    for lag in active_lags:
        working_df[f"{cfg.target}_lag{lag}"] = working_df[cfg.target].shift(lag)

    shifted = working_df[cfg.target].shift(1)
    for window in active_windows:
        working_df[f"{cfg.target}_roll_mean_{window}"] = shifted.rolling(window).mean()
        working_df[f"{cfg.target}_roll_std_{window}"] = shifted.rolling(window).std()
        working_df[f"{cfg.target}_roll_max_{window}"] = shifted.rolling(window).max()
        working_df[f"{cfg.target}_roll_min_{window}"] = shifted.rolling(window).min()

    if len(working_df) > 1:
        working_df[f"{cfg.target}_ewm_mean_24"] = shifted.ewm(span=24, adjust=False).mean()
        working_df[f"{cfg.target}_diff_1"] = shifted.diff(1)
    if len(working_df) > 24:
        working_df[f"{cfg.target}_ewm_mean_72"] = shifted.ewm(span=72, adjust=False).mean()
        working_df[f"{cfg.target}_diff_24"] = shifted.diff(24)

    processed_sections[name] = working_df.dropna().copy()

train_df = processed_sections["train"].copy()
val_processed_full = processed_sections["val_full"].copy()
test_processed_full = processed_sections["test_full"].copy()

val_mask = (val_processed_full.index >= pd.Timestamp(cfg.val_start)) & (val_processed_full.index <= pd.Timestamp(cfg.val_end))
test_mask = test_processed_full.index >= pd.Timestamp(cfg.test_start)

val_df = val_processed_full.loc[val_mask].copy()
test_df = test_processed_full.loc[test_mask].copy()
df_processed = pd.concat([train_df, val_df, test_df], axis=0).sort_index()

print("After feature engineering:", df_processed.shape)
print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)


In [ ]:
target_scaler = StandardScaler()

target_frames = {
    "train": train_df,
    "val": val_df,
    "test": test_df,
    "val_full": val_processed_full,
    "test_full": test_processed_full,
}

forward_targets = {}
for name, frame in target_frames.items():
    values = frame[[cfg.target]].to_numpy(dtype=np.float64) if not frame.empty else np.empty((0, 1))

    if cfg.target_transform in {"log", "sqrt"} and np.any(values < 0):
        raise ValueError(f"Target transform '{cfg.target_transform}' does not support negative values.")

    if cfg.target_transform == "log":
        values = np.log1p(values)
    elif cfg.target_transform == "sqrt":
        values = np.sqrt(values)

    forward_targets[name] = values

y_train = target_scaler.fit_transform(forward_targets["train"])
y_val = target_scaler.transform(forward_targets["val"])
y_test = target_scaler.transform(forward_targets["test"]) if len(forward_targets["test"]) else np.empty((0, 1))
y_val_full = target_scaler.transform(forward_targets["val_full"])
y_test_full = target_scaler.transform(forward_targets["test_full"]) if len(forward_targets["test_full"]) else np.empty((0, 1))

feature_train_df = train_df.drop(columns=[cfg.target]).copy()
num_cols = feature_train_df.select_dtypes(include=[np.number, "bool"]).columns.tolist()
cat_cols = feature_train_df.select_dtypes(include=["object", "category"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ],
    remainder="drop",
)

X_train = np.asarray(preprocessor.fit_transform(feature_train_df), dtype=np.float32)
X_val = np.asarray(preprocessor.transform(val_df.drop(columns=[cfg.target]).copy()), dtype=np.float32)
X_test = np.asarray(preprocessor.transform(test_df.drop(columns=[cfg.target]).copy()), dtype=np.float32) if not test_df.empty else np.empty((0, X_train.shape[1]), dtype=np.float32)
X_val_full = np.asarray(preprocessor.transform(val_processed_full.drop(columns=[cfg.target]).copy()), dtype=np.float32)
X_test_full = np.asarray(preprocessor.transform(test_processed_full.drop(columns=[cfg.target]).copy()), dtype=np.float32) if not test_processed_full.empty else np.empty((0, X_train.shape[1]), dtype=np.float32)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val  :", X_val.shape, "y_val  :", y_val.shape)
print("X_test :", X_test.shape, "y_test :", y_test.shape)


In [ ]:
sequence_inputs = {
    "train": {
        "X": X_train,
        "y": y_train,
        "time_index": train_df.index,
        "start": None,
        "end": None,
    },
    "val": {
        "X": X_val_full,
        "y": y_val_full,
        "time_index": val_processed_full.index,
        "start": cfg.val_start,
        "end": cfg.val_end,
    },
    "test": {
        "X": X_test_full,
        "y": y_test_full,
        "time_index": test_processed_full.index,
        "start": cfg.test_start,
        "end": None,
    },
}

sequence_store = {}

for split_name, payload in sequence_inputs.items():
    X_current = np.asarray(payload["X"])
    y_current = np.asarray(payload["y"])

    if y_current.ndim == 1:
        y_current = y_current.reshape(-1, 1)

    y_flat = y_current.reshape(-1)
    max_start = len(X_current) - cfg.lookback - cfg.horizon + 1

    if max_start <= 0:
        feature_dim = X_train.shape[1]
        X_seq = np.empty((0, cfg.lookback, feature_dim), dtype=np.float32)
        y_seq = np.empty((0, cfg.horizon), dtype=np.float32)
        forecast_times = np.array([], dtype="datetime64[ns]")
    else:
        X_seq_list = []
        y_seq_list = []
        forecast_list = []

        for i in range(max_start):
            X_seq_list.append(X_current[i : i + cfg.lookback])
            y_seq_list.append(y_flat[i + cfg.lookback : i + cfg.lookback + cfg.horizon])
            forecast_list.append(payload["time_index"][i + cfg.lookback])

        X_seq = np.asarray(X_seq_list, dtype=np.float32)
        y_seq = np.asarray(y_seq_list, dtype=np.float32)
        forecast_index = pd.DatetimeIndex(forecast_list)
        mask = np.ones(len(forecast_index), dtype=bool)

        if payload["start"] is not None:
            mask &= forecast_index >= pd.Timestamp(payload["start"])
        if payload["end"] is not None:
            mask &= forecast_index <= pd.Timestamp(payload["end"])

        X_seq = X_seq[mask]
        y_seq = y_seq[mask]
        forecast_times = np.asarray(forecast_index[mask])

    sequence_store[split_name] = {
        "X_seq": X_seq,
        "y_seq": y_seq,
        "times": forecast_times,
    }

X_train_seq = sequence_store["train"]["X_seq"]
y_train_seq = sequence_store["train"]["y_seq"]
train_times = sequence_store["train"]["times"]

X_val_seq = sequence_store["val"]["X_seq"]
y_val_seq = sequence_store["val"]["y_seq"]
val_times = sequence_store["val"]["times"]

X_test_seq = sequence_store["test"]["X_seq"]
y_test_seq = sequence_store["test"]["y_seq"]
test_times = sequence_store["test"]["times"]

if len(X_train_seq) == 0:
    raise ValueError("Khong tao duoc sequence cho train.")
if len(X_val_seq) == 0:
    raise ValueError("Khong tao duoc sequence cho val.")

n_features = X_train_seq.shape[2]

print("Sequence shapes:")
print("Train:", X_train_seq.shape, y_train_seq.shape)
print("Val  :", X_val_seq.shape, y_val_seq.shape)
print("Test :", X_test_seq.shape, y_test_seq.shape)
print("n_features =", n_features)


In [ ]:
artifacts = {
    "cfg": cfg,
    "df": df_processed,
    "train_df": train_df,
    "val_df": val_df,
    "test_df": test_df,
    "X_train": X_train,
    "y_train": y_train,
    "X_val": X_val,
    "y_val": y_val,
    "X_test": X_test,
    "y_test": y_test,
    "X_train_seq": X_train_seq,
    "y_train_seq": y_train_seq,
    "X_val_seq": X_val_seq,
    "y_val_seq": y_val_seq,
    "X_test_seq": X_test_seq,
    "y_test_seq": y_test_seq,
    "train_times": train_times,
    "val_times": val_times,
    "test_times": test_times,
    "target_scaler": target_scaler,
    "preprocessor": preprocessor,
    "n_features": n_features,
}

artifacts.keys()


Neu can inverse du doan tu scale ve gia tri goc, dung `target_scaler.inverse_transform(...)` truoc, sau do ap dung:

- `np.expm1(...)` neu `cfg.target_transform == "log"`
- `np.square(np.clip(..., 0, None))` neu `cfg.target_transform == "sqrt"`
- giu nguyen neu `cfg.target_transform == "none"`
